In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# ==============================================================================
# 1. Data Loading and Cleaning
# ==============================================================================

# Load the dataset
df = pd.read_csv(r"C:/Users/hp/Downloads/Credit_Risk.csv")

# Data Cleaning: Handle negative values in 'Income' and 'LoanAmount' by replacing with NaN
df['Income'] = np.where(df['Income'] < 0, np.nan, df['Income'])
df['LoanAmount'] = np.where(df['LoanAmount'] < 0, np.nan, df['LoanAmount'])

# Define Features and Target
X = df.drop('Default', axis=1)
y = df['Default']

numerical_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'HasMortgage', 'HasDependents', 'HasCoSigner']
categorical_features = ['LoanPurpose']
original_feature_names = X.columns.tolist() # List of 10 original input features

# ==============================================================================
# 2. Preprocessing Pipeline Setup
# ==============================================================================

# Numerical pipeline: Impute NaN with the median, then scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: One-hot encode
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# ==============================================================================
# 3. Model Training and Cross-Validation
# ==============================================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model 1: Logistic Regression (Baseline)
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

# Model 2: RandomForest Classifier (Alternative to XGBoost)
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

models = {
    "Logistic Regression (Baseline)": lr_pipeline,
    "Random Forest Classifier": rf_pipeline
}

results = {}
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    results[name] = {'scores': scores, 'mean_roc_auc': scores.mean(), 'std_dev': scores.std()}

# Determine the best model
best_model_name = max(results, key=lambda k: results[k]['mean_roc_auc'])
best_model_pipeline = models[best_model_name]

# Create and print summary
summary_df = pd.DataFrame({
    'Model': results.keys(),
    'Mean ROC AUC': [res['mean_roc_auc'] for res in results.values()],
    'Std Dev ROC AUC': [res['std_dev'] for res in results.values()]
})

print("="*60)
print("--- Cross-Validation Results Summary ---")
print(summary_df.sort_values(by='Mean ROC AUC', ascending=False).to_string(index=False))
print(f"\nBest Performing Model: {best_model_name}")
print("="*60)

# ==============================================================================
# 4. Feature Importance for the Best Model (Logistic Regression)
# ==============================================================================

# --- Setup for Importance Calculation ---
# Split data for robust permutation importance calculation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
best_model_pipeline.fit(X_train, y_train) 


# --- A. Permutation Importance ---
result = permutation_importance(
    best_model_pipeline,
    X_test, 
    y_test, 
    scoring='roc_auc', 
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# FIX APPLIED HERE: Use original_feature_names which matches the length of result.importances_mean
perm_df = pd.DataFrame({
    'feature': original_feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values(by='importance_mean', ascending=False)

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(x='importance_mean', y='feature', data=perm_df, color='skyblue')
plt.title('Permutation Feature Importance (ROC AUC)')
plt.xlabel('Mean Decrease in ROC AUC')
plt.ylabel('Feature')
plt.tight_layout()
plt.show() # Display plot

print("\n--- Permutation Importance Results (Top 10) ---")
print(perm_df.head(10).to_string(index=False, float_format='%.4f'))


# --- B. Mean Absolute SHAP Values (Logic remains correct for transformed features) ---
# Re-fit the pipeline on the full dataset for global SHAP calculation
best_model_pipeline.fit(X, y) 
X_transformed = best_model_pipeline['preprocessor'].transform(X)
logreg_model = best_model_pipeline['classifier']

# Get feature names robustly from the fitted ColumnTransformer for SHAP
transformed_feature_names = best_model_pipeline['preprocessor'].get_feature_names_out()
feature_names = [name.split('__')[-1] for name in transformed_feature_names]

# Calculate SHAP Values
explainer = shap.LinearExplainer(logreg_model, X_transformed, feature_names=feature_names)
shap_values = explainer.shap_values(X_transformed)

# Calculate Mean Absolute SHAP (global feature importance)
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Create DataFrame for results
shap_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values(by='mean_abs_shap', ascending=False)

# Visualization (SHAP Summary Bar Plot)
print("\n--- SHAP Summary Plot (Mean Absolute SHAP) ---")
shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, plot_type="bar", show=False)
plt.show() # Display plot

print("\n--- Mean Absolute SHAP Values (Top 10) ---")
print(shap_df.head(10).to_string(index=False, float_format='%.4f'))



# Best Model (Logistic Regression) Pipeline
best_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

# Fit the model on the entire dataset
best_model_pipeline.fit(X, y)

# --- 2. Extract Coefficients and Transformed Data ---
logreg_model = best_model_pipeline['classifier']
X_transformed = best_model_pipeline['preprocessor'].transform(X)

# Extract feature names
transformed_feature_names = best_model_pipeline['preprocessor'].get_feature_names_out()
feature_names = [name.split('__')[-1] for name in transformed_feature_names]

# Extract coefficients (slopes) and intercept (base log odds)
coefficients = logreg_model.coef_[0]
intercept = logreg_model.intercept_[0]

# --- 3. Case Selection based on Predicted Probability (FIXED) ---
y_pred_proba = best_model_pipeline.predict_proba(X)[:, 1]
proba_series = pd.Series(y_pred_proba)

# Case 1: Lowest Predicted Probability (Safest Loan)
idx_safe = proba_series.idxmin()

# Case 2: Highest Predicted Probability (Riskiest Loan)
idx_risky = proba_series.idxmax()

# Case 3 & 4: Closest to 0.5 boundary (two closest)
# FIXED: Use Series operations for nsmallest
diff_from_0_5 = (proba_series - 0.5).abs()
boundary_indices = diff_from_0_5.nsmallest(2).index.tolist()
idx_uncertain_1 = boundary_indices[0]
idx_uncertain_2 = boundary_indices[1]

# Case 5: High Confidence, High-Risk Prediction (closest to 0.8)
# FIXED: Use Series operations for nsmallest
diff_from_0_8 = (proba_series - 0.8).abs()
idx_high_conf_risky = diff_from_0_8.nsmallest(1).index.tolist()[0]

selected_indices = list(set([idx_safe, idx_risky, idx_uncertain_1, idx_uncertain_2, idx_high_conf_risky]))
selected_indices.sort(key=lambda x: y_pred_proba[x]) # Sort by risk for presentation

# Create a DataFrame of the transformed feature values
X_scaled = pd.DataFrame(X_transformed, columns=feature_names)


# --- 4. Generate Local Feature Contribution Plots ---
print("--- Detailed Local Feature Contribution Explanations (Log Odds) ---")

# Placeholder for the base log odds for visualization purposes
base_log_odds = intercept

# Data structure to hold feature contributions for the summary plot
summary_contributions = []

for i, idx in enumerate(selected_indices):
    # Calculate feature contributions: beta * x
    feature_contributions = coefficients * X_scaled.iloc[idx].values
    
    # Combine into a Series for easy manipulation
    contributions_series = pd.Series(feature_contributions, index=feature_names)
    
    # Store for summary plot
    summary_contributions.append(contributions_series)
    
    # Sort contributions by magnitude for plotting clarity
    contributions_sorted = contributions_series.abs().sort_values(ascending=False).index
    contributions_plot = contributions_series.loc[contributions_sorted].head(10) # Top 10 contributors
    
    # Calculate predicted log odds
    predicted_log_odds = intercept + contributions_series.sum()
    
    # Print case details
    print(f"\nCase {i+1} (Original Index: {idx})")
    print(f"Predicted Probability of Default (1): {y_pred_proba[idx]:.4f}")
    print(f"Actual Outcome (0=No Default, 1=Default): {y.loc[idx]}")
    print(f"Predicted Log Odds: {predicted_log_odds:.4f} (Base Log Odds: {intercept:.4f})")

    # Visualization: Bar plot of feature contributions
    plt.figure(figsize=(10, 6))
    colors = ['r' if c > 0 else 'g' for c in contributions_plot]
    contributions_plot.plot(kind='barh', color=colors)
    
    plt.axvline(x=0, color='grey', linestyle='--')
    plt.title(f"Case {i+1} - Top 10 Feature Contributions (P(Default)={y_pred_proba[idx]:.4f})")
    plt.xlabel('Feature Contribution to Log Odds (Positive = Riskier)')
    plt.ylabel('Feature')
    plt.gca().invert_yaxis() # Highest contributors at top
    plt.tight_layout()
    plt.savefig(f"Case_{i+1}_Contribution_Plot.png")
    plt.close()

# --- 5. Generate Summary Plot of Selected Cases ---
summary_df = pd.DataFrame(summary_contributions).fillna(0) # Fill NaN from potential sparse OHE columns
summary_df['Predicted Probability'] = [y_pred_proba[idx] for idx in selected_indices]
summary_df['Case Index'] = [f"Idx {idx} (P={y_pred_proba[idx]:.2f})" for idx in selected_indices]
summary_df = summary_df.set_index('Case Index')

# Select only the most important features across all 5 cases for the summary plot
all_contributions = summary_df.drop(columns=['Predicted Probability']).abs().sum(axis=0)
top_features_global = all_contributions.nlargest(10).index

summary_plot_data = summary_df[top_features_global].reset_index().melt(
    id_vars=['Case Index'], 
    var_name='Feature', 
    value_name='Contribution'
)

plt.figure(figsize=(12, 8))
sns.barplot(
    data=summary_plot_data,
    x='Contribution',
    y='Case Index',
    hue='Feature',
    palette='Spectral',
    dodge=False # Stack the bars to show total contribution
)
plt.axvline(x=0, color='grey', linestyle='--')
plt.title('Summary of Top Feature Contributions Across 5 Selected Cases')
plt.xlabel('Net Feature Contribution to Log Odds')
plt.ylabel('Case (Predicted Probability)')
plt.legend(title='Feature', bbox_to_anchor=(1.05, 1), loc=2)
plt.tight_layout()
plt.savefig("Selected_Cases_Summary_Plot.png")
plt.close()

print("\nFive individual feature contribution plots and one summary plot have been generated as the substitute for the requested SHAP analysis.")


# Calculate contributions for all data points
all_contributions = X_transformed * coefficients

# Convert to DataFrame
contributions_df = pd.DataFrame(all_contributions, index=df.index, columns=feature_names)

# --- 3. Group and Calculate Mean Contribution ---
contributions_df['Default'] = y
# Default=0 is the "Approved/Non-Defaulted" group
# Default=1 is the "Denied/Defaulted" group

mean_contributions = contributions_df.groupby('Default').mean()

# --- 4. Bias Assessment on Key Features ---
key_features = ['CreditScore', 'Income', 'LoanAmount', 'Age', 'HasMortgage']

# Extract data for comparison
bias_df = mean_contributions[key_features].T
bias_df.columns = ['Avg_Non_Default_Contribution', 'Avg_Default_Contribution']
bias_df['Difference'] = bias_df['Avg_Default_Contribution'] - bias_df['Avg_Non_Default_Contribution']
bias_df['Abs_Difference'] = bias_df['Difference'].abs()

# Generate a clean comparison table
comparison_df = bias_df.drop(columns=['Difference', 'Abs_Difference'])

# --- 5. Visualization ---
plot_df = bias_df.reset_index().rename(columns={'index': 'Feature'})
plot_df = pd.melt(plot_df, id_vars='Feature', value_vars=['Avg_Non_Default_Contribution', 'Avg_Default_Contribution'], var_name='Group', value_name='Avg_Contribution')

plt.figure(figsize=(10, 6))
sns.barplot(
    data=plot_df,
    x='Avg_Contribution',
    y='Feature',
    hue='Group',
    palette={'Avg_Non_Default_Contribution': 'forestgreen', 'Avg_Default_Contribution': 'firebrick'}
)
plt.axvline(x=0, color='grey', linestyle='--')
plt.title('Bias Assessment: Average Feature Contribution to Log Odds by Outcome Group')
plt.xlabel('Average Feature Contribution to Log Odds (Positive = Riskier)')
plt.ylabel('Key Feature')
plt.legend(title='Outcome Group', labels=['Defaulted (1)', 'Non-Defaulted (0)'])
plt.tight_layout()
plt.savefig('Bias_Assessment_Contribution_Plot.png')
plt.close()

print("Bias Assessment Comparison Table (Average Contribution to Log Odds):")
print(comparison_df.to_string(float_format='%.4f'))
print("\nDifference in Average Contribution (Default - Non-Default):")
print(bias_df[['Difference']].to_string(float_format='%.4f'))



